In [ ]:
from pathlib import Path

root = Path("/kaggle/input")
manifests = list(root.rglob("fixed_partitions.csv"))

print("Saved manifests:")
for p in manifests:
    print(p)

for manifest in manifests:
    base = manifest.parent.parent
    print("\nChecking saved data:", base)

    for domain in ["brackish", "ruod", "norfisk"]:
        folder = base / domain
        images = sum(
            p.is_file() for p in (folder / "images").glob("*")
        )
        labels = sum(
            p.is_file() for p in (folder / "labels").glob("*.txt")
        )
        print(domain, "| readable images:", images, "| labels:", labels)

In [1]:
%pip install -q ultralytics==8.4.207

Note: you may need to restart the kernel to use updated packages.


## ROUD Train


In [ ]:
from pathlib import Path
import json
import hashlib
import shutil
import pandas as pd
import yaml
import torch
from ultralytics import YOLO
from IPython.display import display, FileLink

SOURCE = Path(
    "/kaggle/input/notebooks/farjanaetu/single-data2/lodo_ready"
)
WORK = Path("/kaggle/working")
CONFIG_DIR = WORK / "single_source_configs"
PROJECT = WORK / "single_source_runs"
RUN_NAME = "single_ruod_seed42"

assert torch.cuda.is_available(), "Enable GPU before training."
assert not (PROJECT / RUN_NAME).exists(), (
    "Run folder already exists. Check its checkpoint before restarting."
)

manifest_path = SOURCE / "audit/fixed_partitions.csv"
assert manifest_path.is_file(), f"Missing: {manifest_path}"

manifest = pd.read_csv(manifest_path)
ruod = manifest[
    manifest["domain"].str.lower().eq("ruod")
].copy()

assert len(ruod) == 13999
assert not ruod.duplicated(["domain", "key"]).any()
assert ruod["source_partition"].isin(["train", "val"]).all()

# Replace old working paths with saved-input paths.
# Original filenames and partition assignments remain unchanged.
for column, folder in [
    ("image_path", "images"),
    ("label_path", "labels"),
]:
    ruod[column] = ruod[column].map(
        lambda old: str(
            SOURCE / "ruod" / folder / Path(str(old)).name
        )
    )

    missing = [
        p for p in ruod[column] if not Path(p).is_file()
    ]
    assert not missing, f"Missing {column}: {missing[:5]}"

train = ruod[ruod["source_partition"].eq("train")]
val = ruod[ruod["source_partition"].eq("val")]

assert len(train) == 11219
assert len(val) == 2780
assert set(train["key"]).isdisjoint(set(val["key"]))

if "sha256" in ruod.columns:
    assert set(train["sha256"].dropna()).isdisjoint(
        set(val["sha256"].dropna())
    ), "Hash overlap between train and validation."

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
PROJECT.mkdir(parents=True, exist_ok=True)

train_list = CONFIG_DIR / f"{RUN_NAME}_train.txt"
val_list = CONFIG_DIR / f"{RUN_NAME}_val.txt"

train_list.write_text(
    "\n".join(train["image_path"]) + "\n"
)
val_list.write_text(
    "\n".join(val["image_path"]) + "\n"
)

config_path = CONFIG_DIR / f"{RUN_NAME}.yaml"
config_path.write_text(yaml.safe_dump({
    "path": str(SOURCE),
    "train": str(train_list),
    "val": str(val_list),
    "names": {0: "fish"},
}, sort_keys=False))

# Save experiment records BEFORE training.
ruod.to_csv(
    WORK / "single_ruod_source_partitions.csv",
    index=False,
)

identity = {
    "run": RUN_NAME,
    "training_domain": "ruod",
    "initialization": "COCO-pretrained yolo11n.pt",
    "seed": 42,
    "train_images": len(train),
    "source_validation_images": len(val),
    "validation_role": "Used for checkpoint selection; not independent test",
    "saved_input": str(SOURCE),
    "input_manifest_sha256": hashlib.sha256(
        manifest_path.read_bytes()
    ).hexdigest(),
}
identity_path = WORK / "single_ruod_experiment_identity.json"
identity_path.write_text(json.dumps(identity, indent=2))

print("Train images:", len(train))
print("Source-validation images:", len(val))
print("Reading images directly from saved input.")
print("Starting fresh RUOD-only training...", flush=True)

model = YOLO("yolo11n.pt")

model.train(
    data=str(config_path),
    project=str(PROJECT),
    name=RUN_NAME,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    seed=42,
    deterministic=True,
    patience=30,
    save=True,
    save_period=5,
    plots=True,
    exist_ok=False,
)

run = Path(model.trainer.save_dir)
best = run / "weights/best.pt"
assert best.is_file(), "Best checkpoint missing."

shutil.copy2(config_path, run / "dataset_config.yaml")
shutil.copy2(
    WORK / "single_ruod_source_partitions.csv",
    run / "source_partitions.csv",
)

identity["best_checkpoint_sha256"] = hashlib.sha256(
    best.read_bytes()
).hexdigest()
identity_path.write_text(json.dumps(identity, indent=2))
shutil.copy2(identity_path, run / "experiment_identity.json")

# Individual files for download.
downloads = {
    "single_ruod_best.pt": best,
    "single_ruod_last.pt": run / "weights/last.pt",
    "single_ruod_training_results.csv": run / "results.csv",
    "single_ruod_training_settings.yaml": run / "args.yaml",
    "single_ruod_training_curves.png": run / "results.png",
}

print("\nTRAINING COMPLETE")
for filename, source in downloads.items():
    if source.is_file():
        destination = WORK / filename
        shutil.copy2(source, destination)
        display(FileLink(str(destination)))

display(FileLink(str(WORK / "single_ruod_source_partitions.csv")))
display(FileLink(str(identity_path)))

## test 

In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd
import yaml
import torch
from ultralytics import YOLO
from IPython.display import display, FileLink

SOURCE = Path(
    "/kaggle/input/notebooks/farjanaetu/single-data2/lodo_ready"
)
WEIGHTS = Path(
    "/kaggle/working/single_source_runs/"
    "single_ruod_seed42/weights/best.pt"
)
OUTPUT = Path("/kaggle/working/single_ruod_external_tests")
OUTPUT.mkdir(parents=True, exist_ok=True)

assert WEIGHTS.is_file(), f"Missing checkpoint: {WEIGHTS}"
assert torch.cuda.is_available(), "Enable GPU."

manifest_path = SOURCE / "audit/fixed_partitions.csv"
manifest = pd.read_csv(manifest_path)
model_hash = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()

rows = []

for target, expected_images, expected_boxes in [
    ("brackish", 12444, 14120),
    ("norfisk", 12514, 12514),
]:
    target_df = manifest[
        manifest["domain"].str.lower().eq(target)
    ].copy()

    assert len(target_df) == expected_images
    assert not target_df["key"].duplicated().any()
    assert int(target_df["fish_boxes"].sum()) == expected_boxes

    # Reuse the complete target domain, with saved-input paths.
    target_df["image_path"] = target_df["image_path"].map(
        lambda p: str(SOURCE / target / "images" / Path(str(p)).name)
    )
    target_df["label_path"] = target_df["label_path"].map(
        lambda p: str(SOURCE / target / "labels" / Path(str(p)).name)
    )

    for column in ["image_path", "label_path"]:
        missing = [
            p for p in target_df[column] if not Path(p).is_file()
        ]
        assert not missing, f"Missing {column}: {missing[:3]}"

    image_list = OUTPUT / f"{target}_test.txt"
    image_list.write_text(
        "\n".join(target_df["image_path"]) + "\n"
    )

    config = OUTPUT / f"{target}_test.yaml"
    config.write_text(yaml.safe_dump({
        "path": str(SOURCE),
        "train": str(image_list),
        "val": str(image_list),
        "test": str(image_list),
        "names": {0: "fish"},
    }, sort_keys=False))

    target_df.to_csv(
        OUTPUT / f"single_ruod_to_{target}_image_mapping.csv",
        index=False,
    )

    print(f"\nTesting RUOD-only → {target.upper()}", flush=True)

    metrics = YOLO(str(WEIGHTS)).val(
        data=str(config),
        split="test",
        imgsz=640,
        batch=16,
        device=0,
        workers=2,
        cache=False,
        conf=0.001,
        iou=0.7,
        max_det=300,
        augment=False,
        plots=True,
        save_json=True,
        save_txt=True,
        save_conf=True,
        project=str(OUTPUT),
        name=f"evaluation_{target}",
        exist_ok=False,
    )

    row = {
        "Training domain": "RUOD",
        "Unseen test domain": target,
        "Expected test images": expected_images,
        "Expected fish boxes": expected_boxes,
        "Precision (%)": float(metrics.box.mp) * 100,
        "Recall (%)": float(metrics.box.mr) * 100,
        "mAP50 (%)": float(metrics.box.map50) * 100,
        "mAP50-95 (%)": float(metrics.box.map) * 100,
    }

    csv_path = OUTPUT / f"single_ruod_to_{target}_metrics.csv"
    pd.DataFrame([row]).to_csv(csv_path, index=False)

    details_path = OUTPUT / f"single_ruod_to_{target}_details.json"
    details_path.write_text(json.dumps({
        "model": str(WEIGHTS),
        "model_sha256": model_hash,
        "checkpoint_selection": "RUOD source validation only",
        "training_exclusions": [
            "train_003875", "train_004013", "train_004748",
            "train_006347", "train_007262",
        ],
        "source_validation_exclusions": ["train_008836"],
        "exclusion_reason": "JPEG repair failed on read-only input",
        "metrics": row,
        "evaluation_directory": str(metrics.save_dir),
        "settings": {
            "imgsz": 640,
            "conf": 0.001,
            "nms_iou": 0.7,
            "max_det": 300,
            "augment": False,
        },
    }, indent=2))

    rows.append(row)
    combined = OUTPUT / "single_ruod_external_metrics.csv"
    pd.DataFrame(rows).to_csv(combined, index=False)

    display(pd.DataFrame([row]).round(3))
    display(FileLink(str(csv_path)))
    display(FileLink(str(details_path)))

print("\nBOTH TESTS COMPLETE")
display(pd.DataFrame(rows).round(3))
display(FileLink(str(OUTPUT / "single_ruod_external_metrics.csv")))

## NorFisk-only training

In [2]:
from pathlib import Path
import json
import hashlib
import shutil
import pandas as pd
import yaml
import torch
from ultralytics import YOLO
from IPython.display import display, FileLink

SOURCE = Path(
    "/kaggle/input/notebooks/farjanaetu/single-data2/lodo_ready"
)
WORK = Path("/kaggle/working")
CONFIG_DIR = WORK / "single_source_configs"
PROJECT = WORK / "single_source_runs"
RUN_NAME = "single_norfisk_seed42"
RUN_DIR = PROJECT / RUN_NAME

assert torch.cuda.is_available(), "Enable GPU first."
assert not RUN_DIR.exists(), (
    f"Run already exists: {RUN_DIR}. Check checkpoint before rerunning."
)

manifest_path = SOURCE / "audit/fixed_partitions.csv"
assert manifest_path.is_file(), "Add saved single-data2 output as Input."

manifest = pd.read_csv(manifest_path)
data = manifest[
    manifest["domain"].str.lower().eq("norfisk")
].copy()

assert len(data) == 12514
assert not data.duplicated(["domain", "key"]).any()
assert data["source_partition"].isin(["train", "val"]).all()

# Preserve fixed assignments; update only file paths.
for column, folder in [
    ("image_path", "images"),
    ("label_path", "labels"),
]:
    data[column] = data[column].map(
        lambda old: str(
            SOURCE / "norfisk" / folder / Path(str(old)).name
        )
    )
    missing = [
        p for p in data[column] if not Path(p).is_file()
    ]
    assert not missing, f"Missing {column}: {missing[:3]}"

train = data[data["source_partition"].eq("train")]
val = data[data["source_partition"].eq("val")]

assert len(train) == 10011
assert len(val) == 2503
assert set(train["key"]).isdisjoint(set(val["key"]))

if "sha256" in data.columns:
    assert set(train["sha256"].dropna()).isdisjoint(
        set(val["sha256"].dropna())
    ), "Train/validation hash overlap."

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
PROJECT.mkdir(parents=True, exist_ok=True)

train_list = CONFIG_DIR / f"{RUN_NAME}_train.txt"
val_list = CONFIG_DIR / f"{RUN_NAME}_val.txt"

train_list.write_text("\n".join(train["image_path"]) + "\n")
val_list.write_text("\n".join(val["image_path"]) + "\n")

config_path = CONFIG_DIR / f"{RUN_NAME}.yaml"
config_path.write_text(yaml.safe_dump({
    "path": str(SOURCE),
    "train": str(train_list),
    "val": str(val_list),
    "names": {0: "fish"},
}, sort_keys=False))

partitions_path = WORK / "single_norfisk_source_partitions.csv"
data.to_csv(partitions_path, index=False)

identity = {
    "run": RUN_NAME,
    "training_domain": "norfisk",
    "initialization": "COCO-pretrained yolo11n.pt",
    "seed": 42,
    "planned_train_images": len(train),
    "planned_source_validation_images": len(val),
    "validation_role": "Source validation for checkpoint selection",
    "input_manifest_sha256": hashlib.sha256(
        manifest_path.read_bytes()
    ).hexdigest(),
}

identity_path = WORK / "single_norfisk_experiment_identity.json"
identity_path.write_text(json.dumps(identity, indent=2))

print("NorFisk train:", len(train))
print("NorFisk source validation:", len(val))
print("Fresh initialization; Brackish/RUOD excluded.", flush=True)

model = YOLO("yolo11n.pt")

model.train(
    data=str(config_path),
    project=str(PROJECT),
    name=RUN_NAME,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    seed=42,
    deterministic=True,
    patience=30,
    save=True,
    save_period=5,
    plots=True,
    exist_ok=False,
)

run = Path(model.trainer.save_dir)
best = run / "weights/best.pt"
assert best.is_file(), "Best checkpoint missing."

identity["best_checkpoint_sha256"] = hashlib.sha256(
    best.read_bytes()
).hexdigest()
identity_path.write_text(json.dumps(identity, indent=2))

for source, filename in [
    (partitions_path, "source_partitions.csv"),
    (identity_path, "experiment_identity.json"),
    (config_path, "dataset_config.yaml"),
    (train_list, "source_train.txt"),
    (val_list, "source_val.txt"),
]:
    shutil.copy2(source, run / filename)

downloads = {
    "single_norfisk_best.pt": best,
    "single_norfisk_last.pt": run / "weights/last.pt",
    "single_norfisk_training_results.csv": run / "results.csv",
    "single_norfisk_training_settings.yaml": run / "args.yaml",
    "single_norfisk_training_curves.png": run / "results.png",
}

print("\nNORFISK-ONLY TRAINING COMPLETE")
for filename, source in downloads.items():
    if source.is_file():
        destination = WORK / filename
        shutil.copy2(source, destination)
        display(FileLink(str(destination)))

display(FileLink(str(partitions_path)))
display(FileLink(str(identity_path)))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
NorFisk train: 10011
NorFisk source validation: 2503
Fresh initialization; Brackish/RUOD excluded.
New https://pypi.org/project/ultralytics/8.4.215 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/single_source_configs/single_norfisk_seed42.yaml, degrees=0.0, deterministi

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      2.65G      1.319     0.7409      1.354         26        640: 100% ━━━━━━━━━━━━ 626/626 6.2it/s 1:400.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 79/79 5.5it/s 14.3s0.1s
                   all       2503       2503      0.962      0.964      0.989      0.636

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      2.65G      1.303     0.7091      1.334         23        640: 100% ━━━━━━━━━━━━ 626/626 6.2it/s 1:410.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 79/79 5.5it/s 14.4s0.1s
                   all       2503       2503      0.969       0.97      0.992      0.646

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      2.65G      1.256     0.6599      1.301         20        640: 100% ━━━━━━━━━━━━ 626/626 6.3it/s 1:400.3sss
                 Class     I

/kaggle/working/single_norfisk_best.pt

/kaggle/working/single_norfisk_last.pt

/kaggle/working/single_norfisk_training_results.csv

/kaggle/working/single_norfisk_training_settings.yaml

/kaggle/working/single_norfisk_training_curves.png

/kaggle/working/single_norfisk_source_partitions.csv

/kaggle/working/single_norfisk_experiment_identity.json

## Brackish ও RUOD test

In [3]:
from pathlib import Path
import os
import shutil
import json
import hashlib
import pandas as pd
import yaml
import torch
from ultralytics import YOLO
from IPython.display import display, FileLink

SOURCE = Path(
    "/kaggle/input/notebooks/farjanaetu/single-data2/lodo_ready"
)
WORK = Path("/kaggle/working")
WEIGHTS = WORK / (
    "single_source_runs/single_norfisk_seed42/weights/best.pt"
)
OUTPUT = WORK / "single_norfisk_external_tests"
STAGING = WORK / "single_norfisk_test_data"

assert WEIGHTS.is_file(), f"Checkpoint missing: {WEIGHTS}"
assert torch.cuda.is_available(), "Enable GPU."

OUTPUT.mkdir(parents=True, exist_ok=True)
manifest = pd.read_csv(SOURCE / "audit/fixed_partitions.csv")
model_hash = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()

repair_candidates = {
    "train_003875", "train_004013", "train_004748",
    "train_006347", "train_007262", "train_008836",
}

rows = []

for target, expected_images, expected_boxes in [
    ("brackish", 12444, 14120),
    ("ruod", 13999, 13091),
]:
    data = manifest[
        manifest["domain"].str.lower().eq(target)
    ].copy()

    assert len(data) == expected_images
    assert not data["key"].duplicated().any()
    assert int(data["fish_boxes"].sum()) == expected_boxes

    image_dir = STAGING / target / "images"
    label_dir = STAGING / target / "labels"
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    local_images = []

    for record in data.itertuples(index=False):
        source_image = (
            SOURCE / target / "images" / Path(record.image_path).name
        )
        source_label = (
            SOURCE / target / "labels" / Path(record.label_path).name
        )

        assert source_image.is_file(), str(source_image)
        assert source_label.is_file(), str(source_label)

        image = image_dir / source_image.name
        label = label_dir / source_label.name

        if target == "ruod" and source_image.stem in repair_candidates:
            # Writable copy lets Ultralytics repair these JPEGs.
            if image.is_symlink():
                image.unlink()
            if not image.exists():
                shutil.copy2(source_image, image)
        elif not os.path.lexists(image):
            image.symlink_to(source_image)

        if not os.path.lexists(label):
            label.symlink_to(source_label)

        local_images.append(str(image))

    image_list = OUTPUT / f"{target}_test.txt"
    image_list.write_text("\n".join(local_images) + "\n")

    data["evaluation_image_path"] = local_images
    data.to_csv(
        OUTPUT / f"single_norfisk_to_{target}_image_mapping.csv",
        index=False,
    )

    config = OUTPUT / f"{target}_test.yaml"
    config.write_text(yaml.safe_dump({
        "path": str(STAGING),
        "train": str(image_list),
        "val": str(image_list),
        "test": str(image_list),
        "names": {0: "fish"},
    }, sort_keys=False))

    # Force a fresh label/image scan.
    (label_dir.parent / "labels.cache").unlink(missing_ok=True)

    print(f"\nTesting NORFISK-only → {target.upper()}", flush=True)

    detector = YOLO(str(WEIGHTS))
    metrics = detector.val(
        data=str(config),
        split="test",
        imgsz=640,
        batch=16,
        device=0,
        workers=2,
        cache=False,
        conf=0.001,
        iou=0.7,
        max_det=300,
        augment=False,
        plots=True,
        save_json=True,
        save_txt=True,
        save_conf=True,
        project=str(OUTPUT),
        name=f"evaluation_{target}",
        exist_ok=False,
    )

    evaluated_images = int(detector.validator.seen)

    row = {
        "Training domain": "NorFisk",
        "Unseen test domain": target,
        "Evaluated images": evaluated_images,
        "Expected fish boxes": expected_boxes,
        "Precision (%)": float(metrics.box.mp) * 100,
        "Recall (%)": float(metrics.box.mr) * 100,
        "mAP50 (%)": float(metrics.box.map50) * 100,
        "mAP50-95 (%)": float(metrics.box.map) * 100,
    }

    csv_path = OUTPUT / f"single_norfisk_to_{target}_metrics.csv"
    pd.DataFrame([row]).to_csv(csv_path, index=False)

    details_path = OUTPUT / f"single_norfisk_to_{target}_details.json"
    details_path.write_text(json.dumps({
        "model": str(WEIGHTS),
        "model_sha256": model_hash,
        "checkpoint_selection": "NorFisk source validation only",
        "expected_images": expected_images,
        "evaluated_images": evaluated_images,
        "metrics": row,
        "evaluation_directory": str(metrics.save_dir),
        "writable_JPEG_copies": (
            sorted(repair_candidates) if target == "ruod" else []
        ),
        "settings": {
            "imgsz": 640,
            "conf": 0.001,
            "nms_iou": 0.7,
            "max_det": 300,
            "augment": False,
        },
    }, indent=2))

    rows.append(row)
    combined = OUTPUT / "single_norfisk_external_metrics.csv"
    pd.DataFrame(rows).to_csv(combined, index=False)

    display(pd.DataFrame([row]).round(3))
    display(FileLink(str(csv_path)))
    display(FileLink(str(details_path)))

    assert evaluated_images == expected_images, (
        f"Evaluated {evaluated_images}/{expected_images}. "
        "Keep logs; investigate exclusions before reporting."
    )

print("\nFINAL TWO EXTERNAL TESTS COMPLETE")
display(pd.DataFrame(rows).round(3))
display(FileLink(str(OUTPUT / "single_norfisk_external_metrics.csv")))


Testing NORFISK-only → BRACKISH
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.2 ms, read: 6.9±1.7 MB/s, size: 382.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/working/single_norfisk_test_data/brackish/labels... 12444 images, 6598 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12444/12444 31.2it/s 6:390.0ss
val: New cache created: /kaggle/working/single_norfisk_test_data/brackish/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 778/778 7.8it/s 1:390.1sss
                   all      12444      14120     0.0156     0.0325    0.00354    0.00165
Speed: 0.5ms preprocess, 2.5ms inference, 0.0ms loss, 0.8ms postprocess per image
Saving

AttributeError: 'DetectionModel' object has no attribute 'validator'

In [4]:
from pathlib import Path
import os
import json
import shutil
import hashlib
import pandas as pd
import yaml
from ultralytics import YOLO
from IPython.display import display, FileLink

SOURCE = Path(
    "/kaggle/input/notebooks/farjanaetu/single-data2/lodo_ready"
)
WORK = Path("/kaggle/working")
OUTPUT = WORK / "single_norfisk_external_tests"
WEIGHTS = WORK / (
    "single_source_runs/single_norfisk_seed42/weights/best.pt"
)

OUTPUT.mkdir(parents=True, exist_ok=True)
assert WEIGHTS.is_file()

# Recover the completed Brackish evaluation from memory.
assert "metrics" in globals(), "Session changed; send this message."
assert Path(metrics.save_dir).name == "evaluation_brackish", (
    "metrics is from another evaluation. Send this message."
)

model_hash = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
rows = []

def save_result(target, result, images, boxes):
    row = {
        "Training domain": "NorFisk",
        "Unseen test domain": target,
        "Evaluated images": images,
        "Expected fish boxes": boxes,
        "Precision (%)": float(result.box.mp) * 100,
        "Recall (%)": float(result.box.mr) * 100,
        "mAP50 (%)": float(result.box.map50) * 100,
        "mAP50-95 (%)": float(result.box.map) * 100,
    }

    csv_path = OUTPUT / f"single_norfisk_to_{target}_metrics.csv"
    json_path = OUTPUT / f"single_norfisk_to_{target}_details.json"

    pd.DataFrame([row]).to_csv(csv_path, index=False)
    json_path.write_text(json.dumps({
        "model": str(WEIGHTS),
        "model_sha256": model_hash,
        "checkpoint_selection": "NorFisk source validation only",
        "metrics": row,
        "evaluation_directory": str(result.save_dir),
        "settings": {
            "imgsz": 640,
            "conf": 0.001,
            "nms_iou": 0.7,
            "max_det": 300,
            "augment": False,
        },
    }, indent=2))

    rows.append(row)
    pd.DataFrame(rows).to_csv(
        OUTPUT / "single_norfisk_external_metrics.csv",
        index=False,
    )

    display(pd.DataFrame([row]).round(3))
    display(FileLink(str(csv_path)))
    display(FileLink(str(json_path)))

# Counts confirmed in your completed Brackish log: 0 corrupt.
save_result("brackish", metrics, 12444, 14120)
print("BRACKISH RESULT RECOVERED — no repeat test needed.")

# ---------- RUOD ONLY ----------
manifest = pd.read_csv(SOURCE / "audit/fixed_partitions.csv")
data = manifest[manifest["domain"].str.lower().eq("ruod")].copy()

assert len(data) == 13999
assert not data["key"].duplicated().any()
assert int(data["fish_boxes"].sum()) == 13091

repair_candidates = {
    "train_003875", "train_004013", "train_004748",
    "train_006347", "train_007262", "train_008836",
}

base = WORK / "single_norfisk_test_data/ruod"
image_dir = base / "images"
label_dir = base / "labels"
image_dir.mkdir(parents=True, exist_ok=True)
label_dir.mkdir(parents=True, exist_ok=True)

paths = []

for record in data.itertuples(index=False):
    src_image = SOURCE / "ruod/images" / Path(record.image_path).name
    src_label = SOURCE / "ruod/labels" / Path(record.label_path).name
    assert src_image.is_file(), str(src_image)
    assert src_label.is_file(), str(src_label)

    image = image_dir / src_image.name
    label = label_dir / src_label.name

    if src_image.stem in repair_candidates:
        if image.is_symlink():
            image.unlink()
        if not image.exists():
            shutil.copy2(src_image, image)
    elif not os.path.lexists(image):
        image.symlink_to(src_image)

    if not os.path.lexists(label):
        label.symlink_to(src_label)

    paths.append(str(image))

data["evaluation_image_path"] = paths
data.to_csv(
    OUTPUT / "single_norfisk_to_ruod_image_mapping.csv",
    index=False,
)

image_list = OUTPUT / "ruod_test.txt"
image_list.write_text("\n".join(paths) + "\n")

config = OUTPUT / "ruod_test.yaml"
config.write_text(yaml.safe_dump({
    "path": str(base),
    "train": str(image_list),
    "val": str(image_list),
    "test": str(image_list),
    "names": {0: "fish"},
}, sort_keys=False))

(base / "labels.cache").unlink(missing_ok=True)

# Capture the actual evaluator count through its callback.
observed = {}

def capture_count(validator):
    observed["images"] = int(validator.seen)

detector = YOLO(str(WEIGHTS))
detector.add_callback("on_val_end", capture_count)

print("\nTesting NORFISK-only → RUOD", flush=True)

ruod_metrics = detector.val(
    data=str(config),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    conf=0.001,
    iou=0.7,
    max_det=300,
    augment=False,
    plots=True,
    save_json=True,
    save_txt=True,
    save_conf=True,
    project=str(OUTPUT),
    name="evaluation_ruod",
    exist_ok=False,
)

actual_images = observed.get("images")
assert actual_images is not None, (
    "Evaluation finished, but image count was not captured. "
    "Do not rerun; send the output."
)

save_result("ruod", ruod_metrics, actual_images, 13091)

assert actual_images == 13999, (
    f"Only {actual_images}/13999 images evaluated. Send the log."
)

print("\nFINAL TWO TESTS COMPLETE")
display(pd.DataFrame(rows).round(3))
display(FileLink(str(OUTPUT / "single_norfisk_external_metrics.csv")))

,Training domain,Unseen test domain,Evaluated images,Expected fish boxes,Precision (%),Recall (%),mAP50 (%),mAP50-95 (%)
0,NorFisk,brackish,12444,14120,1.556,3.251,0.354,0.165


/kaggle/working/single_norfisk_external_tests/single_norfisk_to_brackish_metrics.csv

/kaggle/working/single_norfisk_external_tests/single_norfisk_to_brackish_details.json

BRACKISH RESULT RECOVERED — no repeat test needed.

Testing NORFISK-only → RUOD
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 2.2±1.0 MB/s, size: 142.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/working/single_norfisk_test_data/ruod/labels... 13999 images, 10834 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13999/13999 29.3it/s 7:580.1ss
val: /kaggle/working/single_norfisk_test_data/ruod/images/train_003875.jpg: corrupt JPEG restored and saved
val: /kaggle/working/single_norfisk_test_data/ruod/images/train_004013.jpg: corrupt JPEG restored and saved
val: /kaggle/working/single_norfisk_test_data/ruod/images/train_004748.jpg: corrupt JPEG restored and saved
val: /kaggle/working/single_nor

,Training domain,Unseen test domain,Evaluated images,Expected fish boxes,Precision (%),Recall (%),mAP50 (%),mAP50-95 (%)
0,NorFisk,ruod,13999,13091,1.761,5.087,0.183,0.082


/kaggle/working/single_norfisk_external_tests/single_norfisk_to_ruod_metrics.csv

/kaggle/working/single_norfisk_external_tests/single_norfisk_to_ruod_details.json


FINAL TWO TESTS COMPLETE


,Training domain,Unseen test domain,Evaluated images,Expected fish boxes,Precision (%),Recall (%),mAP50 (%),mAP50-95 (%)
0,NorFisk,brackish,12444,14120,1.556,3.251,0.354,0.165
1,NorFisk,ruod,13999,13091,1.761,5.087,0.183,0.082


/kaggle/working/single_norfisk_external_tests/single_norfisk_external_metrics.csv

In [5]:
from pathlib import Path
import json
import hashlib
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display, FileLink

SOURCE = Path(
    "/kaggle/input/notebooks/farjanaetu/single-data2/lodo_ready"
)
EVALUATIONS = Path(
    "/kaggle/working/single_norfisk_external_tests"
)
OUT = Path("/kaggle/working/single_norfisk_error_analysis")
OUT.mkdir(parents=True, exist_ok=True)

CONF = 0.25
THRESHOLDS = [0.50, 0.75]

manifest = pd.read_csv(SOURCE / "audit/fixed_partitions.csv")

def read_yolo(path, prediction=False):
    """Return class + normalized cx,cy,w,h [+ confidence]."""
    columns = 6 if prediction else 5

    if not path.exists():
        if prediction:
            return np.empty((0, columns), dtype=float)
        raise FileNotFoundError(path)

    text = path.read_text().strip()
    if not text:
        return np.empty((0, columns), dtype=float)

    array = np.array([
        [float(x) for x in line.split()]
        for line in text.splitlines() if line.strip()
    ])

    assert array.ndim == 2 and array.shape[1] == columns, path
    assert np.isfinite(array).all(), path
    assert (array[:, 0] == 0).all(), f"Unexpected class: {path}"
    assert (array[:, 3:5] > 0).all(), f"Invalid box: {path}"

    if prediction:
        assert ((array[:, 5] >= 0) & (array[:, 5] <= 1)).all()

    return array

def to_xyxy(boxes):
    result = np.empty_like(boxes)
    result[:, :2] = boxes[:, :2] - boxes[:, 2:] / 2
    result[:, 2:] = boxes[:, :2] + boxes[:, 2:] / 2
    return result

def iou_matrix(pred, truth):
    if len(pred) == 0 or len(truth) == 0:
        return np.zeros((len(pred), len(truth)))

    a, b = to_xyxy(pred), to_xyxy(truth)
    upper_left = np.maximum(a[:, None, :2], b[None, :, :2])
    lower_right = np.minimum(a[:, None, 2:], b[None, :, 2:])
    intersection = np.prod(
        np.maximum(lower_right - upper_left, 0), axis=2
    )
    area_a = np.prod(np.maximum(a[:, 2:] - a[:, :2], 0), axis=1)
    area_b = np.prod(np.maximum(b[:, 2:] - b[:, :2], 0), axis=1)

    union = area_a[:, None] + area_b[None, :] - intersection
    return intersection / np.maximum(union, 1e-12)

def match_predictions(overlap, threshold):
    """
    Predictions are already score-sorted.
    Match each to its highest-IoU eligible unmatched GT.
    """
    used_gt = set()
    matched_gt = set()
    matched_predictions = set()

    for pred_index in range(overlap.shape[0]):
        candidates = [
            gt_index
            for gt_index in range(overlap.shape[1])
            if gt_index not in used_gt
            and overlap[pred_index, gt_index] >= threshold
        ]

        if candidates:
            gt_index = max(
                candidates,
                key=lambda j: overlap[pred_index, j]
            )
            used_gt.add(gt_index)
            matched_gt.add(gt_index)
            matched_predictions.add(pred_index)

    return matched_predictions, matched_gt

def size_group(area):
    if area < 0.01:
        return "small_relative_<1%"
    if area < 0.10:
        return "medium_relative_1_to_<10%"
    return "large_relative_>=10%"

def percentage(numerator, denominator):
    return 100 * numerator / denominator if denominator else np.nan

all_summaries = []
all_size_summaries = []

for target, expected_images, expected_boxes in [
    ("brackish", 12444, 14120),
    ("ruod", 13999, 13091),
]:
    evaluation = EVALUATIONS / f"evaluation_{target}"
    prediction_dir = evaluation / "labels"
    prediction_json = evaluation / "predictions.json"

    assert prediction_json.is_file(), (
        f"Missing saved predictions: {prediction_json}"
    )
    assert prediction_dir.is_dir(), (
        f"Missing prediction TXT folder: {prediction_dir}"
    )

    target_data = manifest[
        manifest["domain"].str.lower().eq(target)
    ].copy()

    assert len(target_data) == expected_images
    assert int(target_data["fish_boxes"].sum()) == expected_boxes

    # Verify TXT predictions against the saved JSON inventory.
    # This prevents silently treating missing prediction files as no detections.
    json_predictions = json.loads(prediction_json.read_text())
    json_counts = {}

    for prediction in json_predictions:
        stem = str(prediction["image_id"])
        json_counts[stem] = json_counts.get(stem, 0) + 1

    expected_stems = {
        Path(str(p)).stem for p in target_data["image_path"]
    }
    assert set(json_counts).issubset(expected_stems), (
        "Prediction JSON image IDs do not match the manifest."
    )

    image_rows = []
    object_rows = []
    ground_truth_backup = {}
    counted_boxes = 0

    for index, record in enumerate(
        target_data.itertuples(index=False), 1
    ):
        stem = Path(record.image_path).stem
        gt_path = (
            SOURCE / target / "labels"
            / Path(record.label_path).name
        )
        pred_path = prediction_dir / f"{stem}.txt"

        gt = read_yolo(gt_path)
        predictions = read_yolo(pred_path, prediction=True)

        assert len(gt) == int(record.fish_boxes), (
            f"Ground-truth count mismatch: {stem}"
        )
        assert len(predictions) == json_counts.get(stem, 0), (
            f"TXT/JSON prediction count mismatch: {stem}"
        )

        ground_truth_backup[stem] = gt_path.read_text()
        counted_boxes += len(gt)

        selected = predictions[predictions[:, 5] >= CONF]
        if len(selected):
            selected = selected[
                np.argsort(-selected[:, 5], kind="stable")
            ]

        overlap = iou_matrix(selected[:, 1:5], gt[:, 1:5])
        matches_by_threshold = {}

        for threshold in THRESHOLDS:
            matched_pred, matched_gt = match_predictions(
                overlap, threshold
            )
            matches_by_threshold[threshold] = matched_gt

            tp = len(matched_gt)
            fp = len(selected) - len(matched_pred)
            fn = len(gt) - tp

            image_rows.append({
                "training_domain": "norfisk",
                "target": target,
                "image_key": stem,
                "group": (
                    "fish_present" if len(gt)
                    else "no_annotated_fish"
                ),
                "confidence": CONF,
                "matching_iou": threshold,
                "GT": len(gt),
                "predictions": len(selected),
                "TP": tp,
                "FP": fp,
                "FN": fn,
            })

        for gt_index, box in enumerate(gt):
            relative_area = float(box[3] * box[4])

            object_rows.append({
                "target": target,
                "image_key": stem,
                "object_index": gt_index,
                "relative_box_area": relative_area,
                "size_group": size_group(relative_area),
                "detected_iou50": (
                    gt_index in matches_by_threshold[0.50]
                ),
                "detected_iou75": (
                    gt_index in matches_by_threshold[0.75]
                ),
                "best_iou_above_confidence": (
                    float(overlap[:, gt_index].max())
                    if len(selected) else 0.0
                ),
            })

        if index % 3000 == 0:
            print(
                f"{target}: analyzed {index}/{expected_images}",
                flush=True,
            )

    assert counted_boxes == expected_boxes

    image_df = pd.DataFrame(image_rows)
    object_df = pd.DataFrame(object_rows)

    image_df.to_csv(
        OUT / f"{target}_per_image_errors.csv", index=False
    )
    object_df.to_csv(
        OUT / f"{target}_per_object_errors.csv", index=False
    )

    summary_rows = []

    for threshold in THRESHOLDS:
        threshold_df = image_df[
            image_df["matching_iou"].eq(threshold)
        ]

        for group in [
            "all", "fish_present", "no_annotated_fish"
        ]:
            subset = (
                threshold_df if group == "all"
                else threshold_df[threshold_df["group"].eq(group)]
            )

            tp = int(subset["TP"].sum())
            fp = int(subset["FP"].sum())
            fn = int(subset["FN"].sum())
            images = len(subset)

            summary_rows.append({
                "training_domain": "norfisk",
                "target": target,
                "confidence": CONF,
                "matching_iou": threshold,
                "group": group,
                "images": images,
                "TP": tp,
                "FP": fp,
                "FN": fn,
                "precision_percent": percentage(tp, tp + fp),
                "recall_percent": percentage(tp, tp + fn),
                "FP_per_image": fp / images if images else np.nan,
                "images_with_FP": int((subset["FP"] > 0).sum()),
            })

    size_rows = []

    for group, subset in object_df.groupby("size_group"):
        size_rows.append({
            "training_domain": "norfisk",
            "target": target,
            "confidence": CONF,
            "size_group": group,
            "GT_boxes": len(subset),
            "TP_iou50": int(subset["detected_iou50"].sum()),
            "TP_iou75": int(subset["detected_iou75"].sum()),
            "recall_iou50_percent": (
                subset["detected_iou50"].mean() * 100
            ),
            "recall_iou75_percent": (
                subset["detected_iou75"].mean() * 100
            ),
        })

    summary = pd.DataFrame(summary_rows)
    size_summary = pd.DataFrame(size_rows)

    summary.to_csv(
        OUT / f"{target}_error_summary.csv", index=False
    )
    size_summary.to_csv(
        OUT / f"{target}_size_summary.csv", index=False
    )

    all_summaries.extend(summary_rows)
    all_size_summaries.extend(size_rows)

    # Preserve annotations needed to reproduce this analysis.
    (OUT / f"{target}_ground_truth.json").write_text(
        json.dumps(ground_truth_backup)
    )

    shutil_path = OUT / f"{target}_predictions.json"
    # Copy saved JSON; no inference.
    import shutil
    shutil.copy2(prediction_json, shutil_path)

    print(f"\n{target.upper()} — ERROR SUMMARY")
    display(summary.round(3))
    print("OBJECT SIZE BREAKDOWN")
    display(size_summary.round(3))

    display(FileLink(str(OUT / f"{target}_error_summary.csv")))
    display(FileLink(str(OUT / f"{target}_size_summary.csv")))

pd.DataFrame(all_summaries).to_csv(
    OUT / "all_targets_error_summary.csv", index=False
)
pd.DataFrame(all_size_summaries).to_csv(
    OUT / "all_targets_size_summary.csv", index=False
)

settings = {
    "training_domain": "norfisk",
    "confidence": CONF,
    "matching_iou": THRESHOLDS,
    "matching": (
        "Score-descending greedy matching to highest-IoU "
        "eligible unmatched ground truth"
    ),
    "relative_area_bins": [
        "[0, 0.01)", "[0.01, 0.10)", "[0.10, 1]"
    ],
    "interpretation": (
        "Descriptive error analysis; not causal attribution. "
        "False positives are relative to retained annotations."
    ),
}

(OUT / "analysis_settings.json").write_text(
    json.dumps(settings, indent=2)
)

archive = Path("/kaggle/working/single_norfisk_analysis_backup.zip")
with zipfile.ZipFile(
    archive, "w", zipfile.ZIP_DEFLATED, allowZip64=True
) as z:
    for path in OUT.iterdir():
        if path.is_file():
            z.write(path, arcname=path.name)

print("\nANALYSIS COMPLETE — no training or inference performed.")
display(FileLink(str(OUT / "all_targets_error_summary.csv")))
display(FileLink(str(OUT / "all_targets_size_summary.csv")))
display(FileLink(str(archive)))

brackish: analyzed 3000/12444
brackish: analyzed 6000/12444
brackish: analyzed 9000/12444
brackish: analyzed 12000/12444

BRACKISH — ERROR SUMMARY


,training_domain,target,confidence,matching_iou,group,images,TP,FP,FN,precision_percent,recall_percent,FP_per_image,images_with_FP
0,norfisk,brackish,0.25,0.50,all,12444,142,3468,13978,3.934,1.006,0.279,3398
1,norfisk,brackish,0.25,0.50,fish_present,5846,142,1247,13978,10.223,1.006,0.213,1191
2,norfisk,brackish,0.25,0.50,no_annotated_fish,6598,0,2221,0,0.000,NaN,0.337,2207
3,norfisk,brackish,0.25,0.75,all,12444,33,3577,14087,0.914,0.234,0.287,3504
4,norfisk,brackish,0.25,0.75,fish_present,5846,33,1356,14087,2.376,0.234,0.232,1297
5,norfisk,brackish,0.25,0.75,no_annotated_fish,6598,0,2221,0,0.000,NaN,0.337,2207


OBJECT SIZE BREAKDOWN


,training_domain,target,confidence,size_group,GT_boxes,TP_iou50,TP_iou75,recall_iou50_percent,recall_iou75_percent
0,norfisk,brackish,0.25,large_relative_>=10%,264,83,28,31.439,10.606
1,norfisk,brackish,0.25,medium_relative_1_to_<10%,3542,49,5,1.383,0.141
2,norfisk,brackish,0.25,small_relative_<1%,10314,10,0,0.097,0.000


/kaggle/working/single_norfisk_error_analysis/brackish_error_summary.csv

/kaggle/working/single_norfisk_error_analysis/brackish_size_summary.csv

... (6 outputs left out)


ANALYSIS COMPLETE — no training or inference performed.


/kaggle/working/single_norfisk_error_analysis/all_targets_error_summary.csv

/kaggle/working/single_norfisk_error_analysis/all_targets_size_summary.csv

/kaggle/working/single_norfisk_analysis_backup.zip

In [6]:
from pathlib import Path
import pandas as pd
from IPython.display import display, FileLink

SOURCE = Path(
    "/kaggle/input/notebooks/farjanaetu/single-data2/lodo_ready"
)
manifest = pd.read_csv(SOURCE / "audit/fixed_partitions.csv")
rows = []

for domain in ["brackish", "ruod", "norfisk"]:
    subset = manifest[
        manifest["domain"].str.lower().eq(domain)
        & manifest["source_partition"].eq("train")
    ]

    counts = {"small_<1%": 0, "medium_1_to_<10%": 0, "large_>=10%": 0}
    expected = int(subset["fish_boxes"].sum())
    total = 0

    for record in subset.itertuples(index=False):
        label = (
            SOURCE / domain / "labels"
            / Path(record.label_path).name
        )
        assert label.is_file(), str(label)

        lines = [
            line.split()
            for line in label.read_text().splitlines()
            if line.strip()
        ]
        assert len(lines) == int(record.fish_boxes)

        for fields in lines:
            assert len(fields) == 5
            cls, cx, cy, width, height = map(float, fields)
            assert cls == 0 and width > 0 and height > 0
            area = width * height

            group = (
                "small_<1%" if area < 0.01
                else "medium_1_to_<10%" if area < 0.10
                else "large_>=10%"
            )
            counts[group] += 1
            total += 1

    assert total == expected

    for group, count in counts.items():
        rows.append({
            "domain": domain,
            "partition": "planned_source_train",
            "train_images": len(subset),
            "size_group": group,
            "fish_boxes": count,
            "share_percent": 100 * count / total,
        })

    print(domain, "completed", flush=True)

table = pd.DataFrame(rows)
output = Path("/kaggle/working/source_train_box_size_distribution.csv")
table.to_csv(output, index=False)

display(table.round(3))
display(FileLink(str(output)))


brackish completed
ruod completed
norfisk completed


,domain,partition,train_images,size_group,fish_boxes,share_percent
0,brackish,planned_source_train,9255,small_<1%,6717,65.500
1,brackish,planned_source_train,9255,medium_1_to_<10%,3296,32.140
2,brackish,planned_source_train,9255,large_>=10%,242,2.360
3,ruod,planned_source_train,11219,small_<1%,6240,58.564
4,ruod,planned_source_train,11219,medium_1_to_<10%,3867,36.293
5,ruod,planned_source_train,11219,large_>=10%,548,5.143
6,norfisk,planned_source_train,10011,small_<1%,0,0.000
7,norfisk,planned_source_train,10011,medium_1_to_<10%,1202,12.007
8,norfisk,planned_source_train,10011,large_>=10%,8809,87.993


/kaggle/working/source_train_box_size_distribution.csv

In [7]:
from pathlib import Path

for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    print(f"\nSEARCHING: {root}", flush=True)

    found = 0
    for path in root.rglob("predictions.json"):
        found += 1
        print(
            path,
            "|",
            round(path.stat().st_size / 1024**2, 2),
            "MB",
            "| prediction TXT folder:",
            (path.parent / "labels").is_dir(),
        )

    print("Prediction JSON files found:", found)


SEARCHING: /kaggle/working
/kaggle/working/single_norfisk_external_tests/evaluation_ruod/predictions.json | 5.17 MB | prediction TXT folder: True
/kaggle/working/single_norfisk_external_tests/evaluation_brackish/predictions.json | 5.99 MB | prediction TXT folder: True
Prediction JSON files found: 2

SEARCHING: /kaggle/input
Prediction JSON files found: 0


In [2]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import display, FileLink

candidates = [Path("/kaggle/working/lodo_ready/brackish/labels")]
candidates += list(
    Path("/kaggle/input").glob("**/lodo_ready/brackish/labels")
)

source = next(
    (p for p in candidates if len(list(p.glob("*.txt"))) == 12444),
    None,
)
assert source is not None, (
    "Complete Brackish labels পাওয়া যায়নি। "
    "Prepared-data notebook output Input হিসেবে add করো।"
)

files = sorted(source.glob("*.txt"))
output = Path("/kaggle/working/brackish_ground_truth_labels.zip")

with ZipFile(output, "w", ZIP_DEFLATED) as archive:
    for file in files:
        archive.write(file, arcname=file.name)

print("Label files:", len(files))
print("ZIP size:", round(output.stat().st_size / 1024**2, 2), "MB")
display(FileLink(str(output)))

Label files: 12444
ZIP size: 2.46 MB


/kaggle/working/brackish_ground_truth_labels.zip